In [11]:
!pip install gcsfs

  Using cached gcsfs-2025.12.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached fsspec-2025.12.0-py3-none-any.whl.metadata (10 kB)
  Using cached google_auth_oauthlib-1.2.3-py3-none-any.whl.metadata (3.1 kB)
  Using cached google_cloud_storage-3.7.0-py3-none-any.whl.metadata (14 kB)
  Using cached google_auth-2.41.1-py2.py3-none-any.whl.metadata (6.6 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached oauthlib-3.3.1-py3-none-any.whl.metadata (7.9 kB)
Using cached google_auth_oauthlib-1.2.3-py3-none-any.whl (19 kB)
Using cached google_auth-2.41.1-py2.py3-none-any.whl (221 kB)
Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl (24 kB)
Using cached oauthlib-3.3.1-py3-none-any.whl (160 kB)
Using cached google_cloud_storage-3.7.0-py3-none-any.whl (303 kB)
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.12.0
    Uninstalling fsspec-2024.12.0:
      Successfully uninstalled fsspec-2024.12.0
  Attempting uninstall:

In [1]:
import gcsfs
from datasets import Dataset

# Read directly with gcsfs
fs = gcsfs.GCSFileSystem()

# Read the parquet file
import pandas as pd
with fs.open("where_you_lora_matters_thesis/datasets/preprocessed_vqa_6k/val/val_shard_0000.parquet", 'rb') as f:
    df = pd.read_parquet(f)

print("Columns:", df.columns.tolist())
print("\nFirst row:")
print(df.iloc[0])

/opt/conda/lib/python3.12/site-packages/google/auth/_default.py:108: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Columns: ['input_ids', 'attention_mask', 'answer', 'pixel_values', 'image_grid_thw']

First row:
input_ids         [151644, 872, 198, 151652, 151655, 151655, 151...
attention_mask    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...
answer                                                         blue
pixel_values      [[-1.0, -0.992, -0.992, -1.0, -1.0, -0.992, -1...
image_grid_thw                                          [1, 36, 26]
Name: 0, dtype: object


In [2]:
from datasets import load_dataset

# Load the actual streaming validation split
ds = load_dataset("HuggingFaceM4/VQAv2", split="validation", streaming=True)

# Skip to sample 6000 (start of your val split)
val_sample = next(iter(ds.skip(20000)))

print("Question:", val_sample['question'])
print("\nAnswers list (10 annotations):", val_sample['answers'])
print("\nMultiple choice answer:", val_sample['multiple_choice_answer'])
print("\nWhat's in your parquet 'answer' field:", "blue")  # From your output

Using custom data configuration default


Question: What color is the sky tinted?

Answers list (10 annotations): [{'answer': 'blue', 'answer_confidence': 'yes', 'answer_id': 1}, {'answer': 'sky blue', 'answer_confidence': 'yes', 'answer_id': 2}, {'answer': 'blue', 'answer_confidence': 'maybe', 'answer_id': 3}, {'answer': 'blue', 'answer_confidence': 'yes', 'answer_id': 4}, {'answer': 'greenish blue', 'answer_confidence': 'yes', 'answer_id': 5}, {'answer': 'green', 'answer_confidence': 'yes', 'answer_id': 6}, {'answer': 'teal', 'answer_confidence': 'maybe', 'answer_id': 7}, {'answer': 'green', 'answer_confidence': 'yes', 'answer_id': 8}, {'answer': 'blue', 'answer_confidence': 'yes', 'answer_id': 9}, {'answer': 'blue', 'answer_confidence': 'yes', 'answer_id': 10}]

Multiple choice answer: blue

What's in your parquet 'answer' field: blue


In [5]:
import pyarrow.parquet as pq

# Load your val shard
table = pq.read_table('gs://where_you_lora_matters_thesis/datasets/preprocessed_vqa_6k/val/val_shard_0000.parquet')

# Check a few answers
for i in range(min(10, len(table))):
    print(f"Sample {i}: {table['answer'][i].as_py()}")

# Load original dataset and verify they match
from datasets import load_dataset
ds = load_dataset("HuggingFaceM4/VQAv2", split="validation", streaming=True)
val_samples = list(ds.skip(20000).take(10))

for i, sample in enumerate(val_samples):
    print(f"Original multiple_choice_answer {i}: {sample['multiple_choice_answer']}")

Sample 0: blue
Sample 1: gray
Sample 2: no
Sample 3: no
Sample 4: no
Sample 5: yes
Sample 6: towards
Sample 7: yes
Sample 8: yes
Sample 9: 5


Using custom data configuration default


Original multiple_choice_answer 0: blue
Original multiple_choice_answer 1: gray
Original multiple_choice_answer 2: no
Original multiple_choice_answer 3: no
Original multiple_choice_answer 4: no
Original multiple_choice_answer 5: yes
Original multiple_choice_answer 6: towards
Original multiple_choice_answer 7: yes
Original multiple_choice_answer 8: yes
Original multiple_choice_answer 9: 5


In [6]:
import pyarrow.parquet as pq
from datasets import load_dataset

# ============ TEST SPLIT ============
print("=== TEST SPLIT ===")
table_test = pq.read_table('gs://where_you_lora_matters_thesis/datasets/preprocessed_vqa_6k/test/test_shard_0000.parquet')

print("Test parquet samples:")
for i in range(min(10, len(table_test))):
    print(f"Sample {i}: {table_test['answer'][i].as_py()}")

print("\nOriginal dataset (starting at sample 6600):")
ds = load_dataset("HuggingFaceM4/VQAv2", split="validation", streaming=True)
test_samples = list(ds.skip(22000).take(10))

for i, sample in enumerate(test_samples):
    print(f"Original multiple_choice_answer {i}: {sample['multiple_choice_answer']}")

# ============ TRAIN SPLIT ============
print("\n\n=== TRAIN SPLIT ===")
print("(Train has answer embedded in input_ids, so we need to decode)")

table_train = pq.read_table('gs://where_you_lora_matters_thesis/datasets/preprocessed_vqa_6k/train/train_shard_0000.parquet')

# Check if train has 'answer' column (shouldn't!)
print(f"Train columns: {table_train.column_names}")
print(f"Has 'answer' column: {'answer' in table_train.column_names}")

# Decode a few input_ids to verify answer is embedded
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-8B-Instruct")

print("\nDecoded train samples (first 3):")
for i in range(min(3, len(table_train))):
    input_ids = table_train['input_ids'][i].as_py()
    decoded = processor.tokenizer.decode(input_ids, skip_special_tokens=False)
    print(f"\n--- Sample {i} ---")
    print(decoded[:500] + "..." if len(decoded) > 500 else decoded)

# Verify against original
print("\nOriginal dataset (starting at sample 0):")
train_samples = list(ds.skip(0).take(3))
for i, sample in enumerate(train_samples):
    print(f"Original Q: {sample['question']}, A: {sample['multiple_choice_answer']}")

=== TEST SPLIT ===
Test parquet samples:
Sample 0: no
Sample 1: meat, grain
Sample 2: no
Sample 3: yes
Sample 4: silver
Sample 5: yes
Sample 6: red
Sample 7: pizza
Sample 8: racks
Sample 9: bakery

Original dataset (starting at sample 6600):


Using custom data configuration default


Original multiple_choice_answer 0: no
Original multiple_choice_answer 1: meat, grain
Original multiple_choice_answer 2: no
Original multiple_choice_answer 3: yes
Original multiple_choice_answer 4: silver
Original multiple_choice_answer 5: yes
Original multiple_choice_answer 6: red
Original multiple_choice_answer 7: pizza
Original multiple_choice_answer 8: racks
Original multiple_choice_answer 9: bakery


=== TRAIN SPLIT ===
(Train has answer embedded in input_ids, so we need to decode)
Train columns: ['input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw']
Has 'answer' column: False


2025-12-12 22:28:53.277558: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765578533.301614   18071 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765578533.311421   18071 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-12 22:28:53.515611: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


ModuleNotFoundError: Could not import module 'AutoProcessor'. Are this object's requirements defined correctly?

In [8]:
import pyarrow.parquet as pq

# Check val shard structure
table = pq.read_table('gs://where_you_lora_matters_thesis/datasets/preprocessed_vqa_6k/val/val_shard_0000.parquet')

print("Columns:", table.column_names)
print("\nFirst row 'answer' field:")
print(table['answer'][0].as_py())
print(f"Type: {type(table['answer'][0].as_py())}")

# Check if it's the list or just the string
if isinstance(table['answer'][0].as_py(), str):
    print("\n❌ Only storing multiple_choice_answer (single string)")
else:
    print("\n✅ Storing full answers list!")

Columns: ['input_ids', 'attention_mask', 'answer', 'pixel_values', 'image_grid_thw']

First row 'answer' field:
blue
Type: <class 'str'>

❌ Only storing multiple_choice_answer (single string)
